# Weight Normalization — A Simple Reparameterization to Accelerate Training of Deep Neural Networks (2016)

_Papers · Foundations & Optimization_

**Split each weight vector into a length and a direction, and train the two separately — so optimization is better conditioned and SGD converges faster.**

---

This notebook is a practice scaffold for the **Weight Normalization — A Simple Reparameterization to Accelerate Training of Deep Neural Networks (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Build a weight-normalized linear layer

Weight normalization rewrites each weight vector as `w = g * v / ||v||` (Eq. 2): a **direction** `v` and a separate **length** scalar `g` per output neuron. The norm is taken over the input dimension (`dim=1`), so every output row gets its own length.

Because `w` is rebuilt from `v` and `g` on every forward pass, the optimizer trains the direction and the length independently — which is exactly the reparameterization the paper proposes.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)

class MyWeightNormLinear(nn.Module):
    """Linear layer with weight normalization from scratch.
       w = g * v / ||v||   — Salimans & Kingma (2016), Eq. 2.
       One length g per output neuron; norm taken over the input dim (dim=1 of the weight)."""
    def __init__(self, in_f, out_f):
        super().__init__()
        self.v    = nn.Parameter(torch.randn(out_f, in_f))   # direction  (per output row)
        self.g    = nn.Parameter(torch.randn(out_f, 1))      # length     (one scalar per neuron)
        self.bias = nn.Parameter(torch.zeros(out_f))

    def weight(self):
        norm = self.v.norm(dim=1, keepdim=True)              # ||v|| per output row
        return self.g * self.v / norm                        # w = g * v / ||v||   (Eq. 2)

    def forward(self, x):                                     # x: (batch, in_f)
        return x @ self.weight().t() + self.bias

### Step 2 — Check it against PyTorch's built-in

To trust our from-scratch layer, we copy the parameters out of PyTorch's official `weight_norm(nn.Linear)` and confirm both compute the **same function** on the same input.

We also verify the key invariant: each row's effective norm `||w||` equals its length `g` — the direction `v`'s own magnitude has been factored out entirely.

In [ ]:
# THE ORACLE: my layer must equal torch's weight_norm(nn.Linear).
ref = nn.Linear(5, 3)
ref = torch.nn.utils.weight_norm(ref, name='weight', dim=0)  # PyTorch's built-in (deprecated alias OK)

mine = MyWeightNormLinear(5, 3)
with torch.no_grad():                       # copy ref's parameters so both compute the same function
    mine.v.copy_(ref.weight_v)
    mine.g.copy_(ref.weight_g)              # ref stores g with shape (out, 1) — matches mine
    mine.bias.copy_(ref.bias)

x = torch.randn(8, 5)
print("allclose(mine, weight_norm(Linear)):", torch.allclose(mine(x), ref(x), atol=1e-6))  # True

# The effective weight's per-row norm equals g:
w = mine.weight()
row_norms = [round(t, 4) for t in w.norm(dim=1).tolist()]
g_values = [round(t, 4) for t in mine.g.squeeze().tolist()]
print("row norms of w:", row_norms)
print("g values     :", g_values)          # should match

### Step 3 — Recompute the worked example and its gradients

With `v=[3,4]` and `g=10`, the unit direction is `[0.6, 0.8]`, so `w = [6, 8]`. We confirm that here.

Then we evaluate the Eq. 3 gradients for `grad_w L = [1,1]`. The crucial property is that `grad_v` comes out **perpendicular** to `v` (their dot product is ~0): updating the direction never changes its length, so the magnitude is controlled solely by `g`.

In [ ]:
# Recompute the worked example: v=[3,4], g=10  ->  w=[6,8].
v = torch.tensor([[3., 4.]])
g = torch.tensor([[10.]])
w_ex = g * v / v.norm(dim=1, keepdim=True)
print("worked w:", w_ex.squeeze().tolist())            # [6.0, 8.0]

# Eq. 3 gradients for grad_w L = [1, 1].
gw = torch.tensor([1., 1.])
vv = torch.tensor([3., 4.])
gg = 10.0
nrm = vv.norm()
grad_g = (gw @ vv) / nrm
grad_v = (gg / nrm) * gw - (gg * grad_g / nrm ** 2) * vv
v_dot_grad_v = round((vv @ grad_v).item(), 6)
print("grad_g:", grad_g.item(), " grad_v:", grad_v.tolist(),
      " v.grad_v:", v_dot_grad_v)               # ~1.4, [0.32,-0.24], ~0 (perpendicular)

### Step 4 — Drop the layer into a small network

Finally we confirm the custom layer composes like any other `nn.Module`: stacked into a 2-layer MLP, it produces the expected output shape. This shows weight normalization is a drop-in replacement for a plain linear layer.

In [ ]:
net = nn.Sequential(MyWeightNormLinear(5, 16), nn.ReLU(), MyWeightNormLinear(16, 2))
out = net(torch.randn(4, 5))
print("net output shape:", out.shape)             # torch.Size([4, 2])

## Visualize the data & results

_Fit the same linear model on the same badly-scaled data, from the identical starting weight, at the same high learning rate — does the weight-normalized reparameterization stay stable while plain gradient descent diverges?_

### Step 1 — Make a badly-scaled dataset

To stress optimization we build a linear-regression problem where one feature is 30x larger than the rest. That makes the loss surface **ill-conditioned**: the ideal step size differs wildly across directions, which is precisely the situation weight normalization is meant to help.

In [ ]:
import numpy as np

def make(seed=0):
    rng = np.random.default_rng(seed)
    N, D = 200, 8
    scales = np.array([1., 1., 1., 1., 1., 1., 1., 30.])   # one feature 30x larger => ill-conditioned
    X = rng.normal(0, 1, (N, D)) * scales
    w_true = rng.normal(0, 1, D)
    y = X @ w_true + rng.normal(0, 0.1, N)
    return X, y

### Step 2 — Train with and without the reparameterization

Both runs start from the *identical* weight and use the same high learning rate. The plain run steps `w` directly. The weight-normalized run keeps `g` and `v` separate and applies the Eq. 3 gradients — note `grad_v` is the self-scaling, perpendicular update that rotates the direction without changing its length.

Decoupling magnitude from direction lets each be updated at its own appropriate effective rate, so the normalized run stays stable while the plain one diverges.

In [ ]:
def train(use_wn, lr=0.003, steps=40):
    X, y = make(0)
    N, D = X.shape
    v_start = np.random.default_rng(7).normal(0, 1, D)   # identical start for both runs
    if use_wn:
        v = v_start.copy()
        g = np.linalg.norm(v)                            # init g = ||v|| so w starts == v_start
    else:
        w = v_start.copy()
    losses = []
    for t in range(steps):
        if use_wn:
            nrm = np.linalg.norm(v)
            w = g * v / nrm                              # w = g*v/||v||  (Eq. 2)
        pred = X @ w
        err = pred - y
        losses.append(round(float(0.5 * np.mean(err ** 2)), 4))
        gw = (X.T @ err) / N                             # grad wrt effective weight w
        if use_wn:
            nrm = np.linalg.norm(v)
            dg = (gw @ v) / nrm                          # Eq. 3: grad_g
            dv = (g / nrm) * gw - (g * dg / nrm ** 2) * v   # Eq. 3: grad_v (perp to v)
            g -= lr * dg
            v -= lr * dv
        else:
            w -= lr * gw
    return losses

wn = train(True)
plain = train(False)
print("weight-normalized final loss:", wn[-1])        # ~2.5891  (stable)
print("plain final loss            :", plain[-1])     # ~9.3e20  (diverged)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A neuron has direction $\mathbf{v}=[0,\,5,\,12]$ and length $g=26$. Compute the reparameterized weight $\mathbf{w}$ and verify $\lVert\mathbf{w}\rVert=g$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Length of the direction: $\lVert\mathbf{v}\rVert=\sqrt{0^2+5^2+12^2}=\sqrt{0+25+144}=\sqrt{169}=13$. — _Need the unit direction, so divide by this._
- Weight: $\mathbf{w}=g\,\mathbf{v}/\lVert\mathbf{v}\rVert=26\cdot[0,5,12]/13=2\cdot[0,5,12]=[0,10,24]$. — _Scale the unit direction by $g$._
- Magnitude: $\lVert\mathbf{w}\rVert=\sqrt{0^2+10^2+24^2}=\sqrt{0+100+576}=\sqrt{676}=26$. — _Confirms the design: the length equals $g$._

**Answer:** $\mathbf{w}=[0,\,10,\,24]$ and $\lVert\mathbf{w}\rVert=26=g$. Doubling or halving $\mathbf{v}$ would not change $\mathbf{w}$ at all — only its direction matters.

</details>

**Problem 2.** With $\mathbf{v}=[3,4]$, $g=10$, and $\nabla_{\mathbf{w}}L=[1,1]$, the worked example gave $\nabla_g L=1.4$ and $\nabla_{\mathbf{v}}L=[0.32,-0.24]$. After one gradient-descent step with learning rate $0.1$, what are the new $g$ and $\mathbf{v}$, and did the direction's length change?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Update $g$: $g \leftarrow g - 0.1\cdot\nabla_g L = 10 - 0.1\cdot1.4 = 9.86$. — _Plain gradient step on the length scalar._
- Update $\mathbf{v}$: $\mathbf{v}\leftarrow[3,4]-0.1\cdot[0.32,-0.24]=[2.968,\,4.024]$. — _Plain gradient step on the direction._
- New length of $\mathbf{v}$: $\sqrt{2.968^2+4.024^2}=\sqrt{8.809+16.193}\approx\sqrt{25.00}\approx5.0002$. — _Because the update was perpendicular to $\mathbf{v}$, its length is unchanged to first order._

**Answer:** $g=9.86$, $\mathbf{v}\approx[2.968,4.024]$. The length of $\mathbf{v}$ stays $\approx5$ — the perpendicular update rotated the direction without lengthening it, exactly as the gradient algebra predicts. The magnitude of the effective weight is controlled solely by $g$ (now $9.86$).

</details>

**Problem 3.** Ablation. In the CODEVIZ both runs start from the identical weight and use the same high learning rate, but the plain (unnormalized) run diverges while the weight-normalized run stays stable. Why does the reparameterization prevent the blow-up?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the data has one feature on a much larger scale than the others, so the loss surface is badly conditioned — the right step size differs wildly across directions. — _This is the conditioning problem weight norm targets._
- In the plain run, one large learning rate has to serve both the magnitude and the direction of $\mathbf{w}$; for the steep direction it is far too big, so the weight overshoots and the loss explodes. — _Magnitude and direction are tangled in one vector._
- In the weight-normalized run, the direction update is automatically scaled by $g/\lVert\mathbf{v}\rVert$ and projected perpendicular to $\mathbf{v}$, and the magnitude is handled by its own scalar $g$. Neither piece overshoots. — _Decoupling lets each be updated at its own appropriate effective rate._

**Answer:** Plain gradient descent diverges (loss past $10^6$ within a few steps in our run) because a single learning rate is wrong for the tangled magnitude+direction. Weight normalization decouples them: the self-scaling, perpendicular update on $\mathbf{v}$ plus the separate $g$ keeps the step well-sized, so the loss descends steadily and stays bounded. This reproduces the paper's qualitative claim that the reparameterization improves conditioning and lets you train stably at higher learning rates.

</details>